In [ ]:
# ============================================================
# INDEPENDENT BASELINE MODELS
# ResNet18, ResNet50, VGG16, DenseNet121, InceptionV3,
# EfficientNet-B0, MobileNetV3-Large, ConvNeXt-Tiny,
# ViT-B/16, Swin-Tiny
# ============================================================
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from tqdm.auto import tqdm
import timm
import warnings
warnings.filterwarnings("ignore")

# -------------------- CONFIG --------------------
BASE_CFG = {
    "batch_size": 32,
    "epochs": 40,
    "lr": 2e-4,
    "warmup_epochs": 2,
    "dropout": 0.3,
    "label_smoothing": 0.1,
    "weight_decay": 1e-4,
    "test_size": 0.20,       
    "val_size": 0.15,        
    "seed": 42,
}

BASELINE_MODELS = {
    "ResNet18":          "resnet18",
    "ResNet50":          "resnet50",
    "VGG16":             "vgg16",
    "DenseNet121":       "densenet121",
    "InceptionV3":       "inception_v3",
    "EfficientNet-B0":   "efficientnet_b0",
    "MobileNetV3-Large": "mobilenetv3_large_100",
    "ConvNeXt-Tiny":     "convnext_tiny",
    "ViT-B_16":          "vit_base_patch16_224",
    "Swin-Tiny":         "swin_tiny_patch4_window7_224",
}

# -------------------- INDEPENDENT SPLIT --------------------
def make_independent_split():
    """Create train / val / test indices independently (no folds used)"""
    idx = np.arange(len(y))
    # first split → trainval / test
    trainval_idx, test_idx = train_test_split(
        idx, test_size=BASE_CFG["test_size"],
        stratify=y, random_state=BASE_CFG["seed"]
    )
    # second split → train / val
    train_idx, val_idx = train_test_split(
        trainval_idx, test_size=BASE_CFG["val_size"],
        stratify=y[trainval_idx], random_state=BASE_CFG["seed"]
    )
    print(f"Independent Split → Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")
    return train_idx, val_idx, test_idx

# -------------------- MODEL --------------------
class BaselineNet(nn.Module):
    def __init__(self, model_name, num_classes=4, pretrained=True, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(
            model_name, pretrained=pretrained,
            num_classes=0, global_pool="avg"
        )
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(self.backbone.num_features, num_classes)

    def forward(self, x):
        return self.fc(self.drop(self.backbone(x)))

def build_baseline(model_name):
    return BaselineNet(model_name, num_classes=len(CLASSES),
                       pretrained=True, dropout=BASE_CFG["dropout"]).to(device)

# -------------------- DATALOADER --------------------
def make_loader(idx, tf, shuffle):
    return DataLoader(
        DentalDS(X, y, idx, tf),
        batch_size=BASE_CFG["batch_size"],
        shuffle=shuffle,
        num_workers=0,
        pin_memory=True,
        drop_last=shuffle
    )

# -------------------- EVAL + PER-CLASS --------------------
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    P, Y = [], []
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, enabled=use_amp):
            out = model(xb)
        P.append(out.float().softmax(1).cpu())
        Y.append(yb)
    P = torch.cat(P).numpy()
    Y = torch.cat(Y).numpy()
    return P, Y, P.argmax(1)

def per_class_metrics(y_true, y_pred, class_names):
    cm = confusion_matrix(y_true, y_pred)
    rows = []
    total = cm.sum()
    for i, name in enumerate(class_names):
        TP = cm[i, i]
        FN = cm[i, :].sum() - TP
        FP = cm[:, i].sum() - TP
        TN = total - (TP + FN + FP)
        acc  = (TP + TN) / total * 100
        tpr  = TP / (TP + FN + 1e-8) * 100
        fnr  = FN / (TP + FN + 1e-8) * 100
        fpr  = FP / (FP + TN + 1e-8) * 100
        tnr  = TN / (TN + FP + 1e-8) * 100
        prec = TP / (TP + FP + 1e-8) * 100
        f1   = 2 * prec * tpr / (prec + tpr + 1e-8)
        err  = 100 - acc
        rows.append({
            "Class": name,
            "Accuracy (%)": round(acc, 2),
            "TPR (%)": round(tpr, 2),
            "FNR (%)": round(fnr, 2),
            "FPR (%)": round(fpr, 2),
            "TNR (%)": round(tnr, 2),
            "Precision (%)": round(prec, 2),
            "F1 Score (%)": round(f1, 2),
            "Error Rate (%)": round(err, 2)
        })
    return pd.DataFrame(rows)

def print_per_class_table(df, model_name):
    cols = ["Class", "Accuracy (%)", "TPR (%)", "FNR (%)", "FPR (%)",
            "TNR (%)", "Precision (%)", "F1 Score (%)", "Error Rate (%)"]
    print(f"\n===== {model_name} | Per-Class Results =====")
    print("| " + " | ".join(cols) + " |")
    print("|" + "|".join(["-" * (len(c)+2) for c in cols]) + "|")
    for _, row in df.iterrows():
        print("| " + " | ".join([str(row[c]) for c in cols]) + " |")

# -------------------- TRAIN ONE MODEL --------------------
def train_one_model(display_name, model_name, train_idx, val_idx, test_idx):
    print(f"\n{'='*70}")
    print(f"🚀 Independent Run → {display_name}")
    print(f"{'='*70}")

    tr_loader = make_loader(train_idx, train_tf, True)
    va_loader = make_loader(val_idx,   eval_tf, False)
    te_loader = make_loader(test_idx,  eval_tf, False)

    cnt = np.bincount(y[train_idx], minlength=len(CLASSES))
    w = torch.tensor(len(train_idx) / (len(CLASSES) * cnt + 1e-6),
                     dtype=torch.float32).to(device)
    criterion = nn.CrossEntropyLoss(weight=w, label_smoothing=BASE_CFG["label_smoothing"])

    model = build_baseline(model_name)
    params = sum(p.numel() for p in model.parameters()) / 1e6

    opt = torch.optim.AdamW(model.parameters(), lr=BASE_CFG["lr"],
                            weight_decay=BASE_CFG["weight_decay"])
    sched = torch.optim.lr_scheduler.SequentialLR(opt, [
        torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.1,
                                          total_iters=BASE_CFG["warmup_epochs"]),
        torch.optim.lr_scheduler.CosineAnnealingLR(
            opt, T_max=max(BASE_CFG["epochs"] - BASE_CFG["warmup_epochs"], 1))
    ], milestones=[BASE_CFG["warmup_epochs"]])
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    best_f1 = -1.0
    ckpt = os.path.join(OUT_DIR, f"indep_{display_name}.pth")

    for ep in range(BASE_CFG["epochs"]):
        model.train()
        run_loss, n = 0.0, 0
        for xb, yb in tqdm(tr_loader, desc=f"{display_name} Ep {ep+1}", leave=False):
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                loss = criterion(model(xb).float(), yb)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            run_loss += loss.item() * len(yb)
            n += len(yb)
        sched.step()

        _, labels, preds = evaluate(model, va_loader)
        val_f1 = f1_score(labels, preds, average="macro")
        flag = ""
        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), ckpt)
            flag = " ✅"
        print(f"Ep {ep+1:02d} | loss {run_loss/n:.4f} | val_macroF1 {val_f1:.4f}{flag}")

    # Test
    model.load_state_dict(torch.load(ckpt, map_location=device))
    probs, labels, preds = evaluate(model, te_loader)

    acc = accuracy_score(labels, preds) * 100
    macro_f1 = f1_score(labels, preds, average="macro") * 100

    per_class_df = per_class_metrics(labels, preds, CLASSES)
    print_per_class_table(per_class_df, display_name)
    per_class_df.to_csv(os.path.join(OUT_DIR, f"per_class_{display_name}.csv"), index=False)

    print(f"\n✅ {display_name} finished | Acc: {acc:.2f}% | Macro-F1: {macro_f1:.2f}% | Params: {params:.2f}M")
    return {
        "Model": display_name,
        "Accuracy (%)": round(acc, 2),
        "Macro-F1 (%)": round(macro_f1, 2),
        "Params (M)": round(params, 2)
    }, per_class_df

# ============================================================
# RUN ALL MODELS INDEPENDENTLY
# ============================================================
os.makedirs(OUT_DIR, exist_ok=True)

# Create one independent split
train_idx, val_idx, test_idx = make_independent_split()

all_summaries = []
all_per_class = []

for display_name, model_name in BASELINE_MODELS.items():
    try:
        summary, per_df = train_one_model(display_name, model_name,
                                          train_idx, val_idx, test_idx)
        all_summaries.append(summary)
        all_per_class.append(per_df)
    except Exception as e:
        print(f"\n❌ {display_name} failed → {e}")
        continue

# -------------------- FINAL SUMMARY --------------------
summary_df = pd.DataFrame(all_summaries)
print("\n" + "="*80)
print("INDEPENDENT MODELS - OVERALL SUMMARY")
print("="*80)
print(summary_df.to_string(index=False))
summary_df.to_csv(os.path.join(OUT_DIR, "independent_summary.csv"), index=False)

combined = pd.concat(all_per_class, ignore_index=True)
combined.to_csv(os.path.join(OUT_DIR, "independent_per_class_all.csv"), index=False)

print(f"\n✅ all models finished")
print(f"✅ Results → {OUT_DIR}")